# Multi-Free Field Analysis Example using DAPI

This example shows how to run OpenSeesMP in DesignSafe from a jupyter notebook using the DesignSafe API (dapi).

A set of four 1D profiles is analyzed using OpenSeesMP.

<img src = "https://github.com/DesignSafe-CI/dapi/blob/main/examples/opensees/multi-freeField.png?raw=true.png"  height="400" width="400" align = "center">

# Setup DAPI and start OpenSeesMP job

Docs: [Authentication](https://designsafe-ci.github.io/dapi/authentication) covers credentials from the environment, a `.env` file, or prompts.

In [ ]:
%pip install --quiet --upgrade dapi

In [ ]:
%matplotlib inline

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "./DS_input"))

from plotAcc import plot_acc

### Setup job description

Docs: [Path translation](https://designsafe-ci.github.io/dapi/files#path-translation) and [Job Submission](https://designsafe-ci.github.io/dapi/jobs#job-submission), including every [`generate()` parameter](https://designsafe-ci.github.io/dapi/jobs#generate-parameters).

In [ ]:
# Import DAPI and other required libraries

import json
import os

from dapi import DSClient  # noqa: F401  (your answer uses it)

In [ ]:
ds = ...  # TODO: create the authenticated client

In [ ]:
ds_path = os.getcwd() + "/DS_input"
print(f"Input directory: {ds_path}")

try:
    input_uri = ...  # TODO: translate ds_path (works on DesignSafe JupyterHub)
except ValueError:
    # anywhere else: dapi uploads the folder to MyData once and stages from there
    input_uri = ...  # TODO: prepare_inputs, then translate its staged_dir

print(f"Input URI: {input_uri}")

In [ ]:
# Job configuration parameters
jobname: str = "opensees-MP-multiMotion-dapi"
app_id: str = "opensees-mp-s3"
input_filename: str = "Main_multiMotion.tcl"
control_exec_Dir: str = "DS_input"  # Folder with files including input_filename
tacc_allocation: str = "DS-Portal-SPARC2026"  # <-- replace with your allocation
archive_system: str = "designsafe"
queue: str = "skx-dev"  # or "debug"
control_nodeCount: int = 1
control_corespernode: int = 16
max_job_minutes: int = 10

In [ ]:
# Generate job request dictionary using app defaults
job_dict = ...  # TODO: ds.jobs.generate with the parameters above

In [ ]:
# Customize job settings
job_dict["name"] = jobname
job_dict["nodeCount"] = control_nodeCount
job_dict["coresPerNode"] = control_corespernode

print("Generated job request:")
print(json.dumps(job_dict, indent=2, default=str))

### Run job

Docs: [Job Monitoring](https://designsafe-ci.github.io/dapi/jobs#job-monitoring) and [Job Analysis](https://designsafe-ci.github.io/dapi/jobs#job-analysis).

In [ ]:
# Submit job using dapi
submitted_job = ...  # TODO
print(f"Job launched with UUID: {submitted_job.uuid}")

In [ ]:
# Monitor job status using dapi
final_status = ...  # TODO: poll to completion

print(f"Job finished with status: {final_status}")

# TODO: interpret the final status

# TODO: display the runtime summary

In [ ]:
# Print out last message from the job
...  # TODO

# Postprocess Results

Docs: [Output Management](https://designsafe-ci.github.io/dapi/jobs#output-management) and [Download](https://designsafe-ci.github.io/dapi/files#download).

### Identify job and archived location

In [ ]:
# Get archive information using dapi
archive_uri = submitted_job.archive_uri
print(f"Archive URI: {archive_uri}")

# List archive contents
archive_files = ds.files.list(archive_uri)
print("\nArchive contents:")
for item in archive_files:
    print(f"- {item.name} ({item.type})")

### Go to archived folder

In [ ]:
# Download the inputDirectory folder which contains results
input_dir_archive_uri = f"{archive_uri}/inputDirectory"
try:
    # List contents of inputDirectory in archive
    input_dir_files = ds.files.list(input_dir_archive_uri)
    print("\nFiles in inputDirectory:")
    for item in input_dir_files:
        print(f"- {item.name} ({item.type})")

except Exception as e:
    print(f"Error accessing archive: {e}")

### Plot acceleration response spectra

Plot acceleration response spectra on log-linear scale

In [ ]:
# A directory to plot from. On DesignSafe JupyterHub the archive is a
# local folder; anywhere else, download the recorder outputs first.
archive_path = ds.files.to_path(input_dir_archive_uri)
if not os.path.isdir(archive_path):
    archive_path = "archive_outputs"
    os.makedirs(archive_path, exist_ok=True)
    for item in ds.files.list(input_dir_archive_uri):
        if item.name.endswith(".out"):
            ds.files.download(
                f"{input_dir_archive_uri}/{item.name}",
                f"{archive_path}/{item.name}",
            )
print(archive_path)

In [ ]:
# Plot directly from the archive path — no os.chdir needed
plot_acc(archive_path)